# TrackNet V2: inspeção e validação

Configure os caminhos e as resoluções abaixo e execute as células em ordem, em um kernel novo.
O notebook importa as implementações do projeto; não treina um modelo.
Um checkpoint com `state_dict` é obrigatório antes de qualquer previsão ou validação.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "models" / "TracknetV2.py").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "models" / "TracknetV2.py").is_file():
    raise FileNotFoundError("Abra o notebook a partir da raiz do projeto ou de Notebooks/.")
sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from datasets.TracknetV2Dataset import TracknetV2Dataset
from models.TracknetV2 import TrackentV2
from utils.TracknetV2.loss import wbce_loss
from utils.TracknetV2.train_val import validate
from utils.helpers import make_prediction_grid, show_grid_batches


## Configuração

Informe a pasta que contém `frames_out/`, `gts/` e `labels/`, organizados por partida e clip.
As coordenadas dos labels usam a resolução original. Os GTs devem estar na resolução de entrada;
altura e largura de entrada precisam ser múltiplos de 8. Threshold e tolerância mantêm o protocolo do projeto.

In [ ]:
DATASET_PATH = PROJECT_ROOT / "assets" / "val"
CHECKPOINT_PATH = None  # Defina um arquivo de pesos; não são usados pesos aleatórios.
ORIGINAL_WIDTH = 1920
ORIGINAL_HEIGHT = 1080
INPUT_WIDTH = 640
INPUT_HEIGHT = 360
BATCH_SIZE = 2
THRESHOLD = 0.5
MAX_DISTANCE = 10


In [ ]:
if CHECKPOINT_PATH is None:
    raise ValueError("Defina CHECKPOINT_PATH com um state_dict treinado antes de prever ou validar.")
checkpoint_path = Path(CHECKPOINT_PATH)
if not checkpoint_path.is_absolute():
    checkpoint_path = PROJECT_ROOT / checkpoint_path
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Checkpoint não encontrado: {checkpoint_path}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TrackentV2(mode="train").to(device)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval()
criterion = wbce_loss


In [ ]:
dataset = TracknetV2Dataset(
    frames_path=str(DATASET_PATH / "frames_out"),
    gts_path=str(DATASET_PATH / "gts"),
    labels_path=str(DATASET_PATH / "labels"),
    original_width=ORIGINAL_WIDTH,
    original_height=ORIGINAL_HEIGHT,
    input_width=INPUT_WIDTH,
    input_height=INPUT_HEIGHT,
)
if len(dataset) == 0:
    raise ValueError(f"Dataset sem janelas temporais: {DATASET_PATH}")
val_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Janelas: {len(dataset)} | Dispositivo: {device}")


## Dados: frame central e GT correspondente

Linha superior: RGB central de cada janela. Linha inferior: GT central.
Inputs, GTs e labels seguem a ordem temporal `[t-2, t-1, t]`.

In [ ]:
grid = show_grid_batches(val_loader, batch_size=BATCH_SIZE)
plt.figure(figsize=(12, 6))
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.show()


## Previsão do checkpoint

O grid compara o frame RGB central com a probabilidade do mesmo canal temporal.
O helper respeita o modo explícito de saída do modelo e não modifica os inputs.

In [ ]:
inputs, _, _ = next(iter(val_loader))
prediction_grid = make_prediction_grid(model, inputs.to(device))
plt.figure(figsize=(12, 6))
plt.imshow(prediction_grid.permute(1, 2, 0).cpu().numpy())
plt.axis("off")
plt.show()


## Validação

Reutiliza a loss, o pós-processamento e as contagens atuais, incluindo falsos positivos em frames sem bola.
O protocolo agrega os três canais de janelas sobrepostas; a inferência em vídeo usa apenas o canal atual.
Não compare esse resultado com outro split ou protocolo sem documentar as diferenças.

In [ ]:
val_loss, precision, recall, f1_score = validate(
    model, val_loader, criterion, device,
    max_distance=MAX_DISTANCE, threshold=THRESHOLD,
)
print(f"Loss: {val_loss:.6f} | Precisão: {precision:.4f} | Recall: {recall:.4f} | F1: {f1_score:.4f}")
